In [0]:
from pyspark.sql import functions as F

bronze_order_items = spark.table("brazilian_ecommerce.prod.bronze_order_items")

staging_order_items = (
    bronze_order_items
    .withColumn("order_item_id", F.col("order_item_id").cast("int"))
    .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date"))
    .withColumn("price", F.col("price").cast("decimal(10,2)"))
    .withColumn("freight_value", F.col("freight_value").cast("decimal(10,2)"))
    .filter(F.col("order_id").isNotNull() & F.col("order_item_id").isNotNull())
    .dropDuplicates(["order_id", "order_item_id"])
    .withColumn("_loaded_at", F.current_timestamp())
)

full_table_name = "brazilian_ecommerce.prod.staging_order_items"

(
    staging_order_items.write
    .mode("overwrite")
    .option("comment", "order items staging table")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

# composite primary key
spark.sql(f"""
    ALTER TABLE {full_table_name}
    ALTER COLUMN order_id SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ALTER COLUMN order_item_id SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT pk_staging_order_items PRIMARY KEY (order_id, order_item_id)
""")

# foreign keys
spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT fk_staging_order_items_order_id
    FOREIGN KEY (order_id)
    REFERENCES brazilian_ecommerce.prod.staging_orders (order_id)
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT fk_staging_order_items_product_id
    FOREIGN KEY (product_id)
    REFERENCES brazilian_ecommerce.prod.staging_products (product_id)
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT fk_staging_order_items_seller_id
    FOREIGN KEY (seller_id)
    REFERENCES brazilian_ecommerce.prod.staging_sellers (seller_id)
""")

print(f"Loaded {staging_order_items.count()} rows into {full_table_name}")